# Fine-tune the receipt student

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --no-deps git+https://github.com/sourav-ydv/receipt-router.git

In [ ]:
import os
print(os.listdir("/kaggle/input"))

## Settings

In [ ]:
import json
import shutil

DATA_DIR = "/kaggle/input/receipt-sft"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_FILE = "train_mixed.jsonl"
RUN = "mixed"
MAX_LEN = 2048
EPOCHS = 3
SEED = 42

## Model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

## Data

In [ ]:
from datasets import load_dataset


def read(name):
    return load_dataset("json", data_files=f"{DATA_DIR}/{name}", split="train")


def to_text(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False)}


train = read(TRAIN_FILE).map(to_text)
valid = read("validation_idr.jsonl").map(to_text)

lengths = [len(tokenizer(t)["input_ids"]) for t in train["text"]]
print(len(train), len(valid), max(lengths), sum(n > MAX_LEN for n in lengths))
print(train[0]["text"])

## Train

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train,
    eval_dataset=valid,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="no",
        seed=SEED,
        report_to="none",
        output_dir="outputs",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()
print(stats.metrics)

## Evaluate on the test split

In [ ]:
import torch

from receipts.formatting import instruction_for
from receipts.schema import Receipt
from receipts.scoring import score, summarize
from receipts.validators import check, parse_receipt

with open(f"{DATA_DIR}/test.jsonl", encoding="utf-8") as f:
    test = [json.loads(line) for line in f if line.strip()]
test = [row for row in test if row["usable"]]

FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"


def generate(texts, batch_size=4):
    outputs = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": instruction_for("idr")},
                    {"role": "user", "content": text},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            for text in batch
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=800, do_sample=False)
        new_tokens = generated[:, inputs["input_ids"].shape[1]:]
        outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    return outputs


raw = generate([row["text"] for row in test])

In [ ]:
everything = []
accepted = []
records = []

for row, text in zip(test, raw):
    gold = Receipt(**row["label"])
    pred = parse_receipt(text)
    problems = ["unparseable"] if pred is None else check(pred)
    result = score(pred or Receipt(), gold)
    everything.append(result)
    if not problems:
        accepted.append(result)
    records.append({"raw": text, "problems": problems})


def show(name, scores):
    summary = summarize(scores)
    cells = ", ".join(f"{key} {value:.1%}" for key, value in summary.items())
    print(f"{name} ({len(scores)}): {cells}")


unparseable = sum(1 for r in records if r["problems"] == ["unparseable"])
print("unparseable:", unparseable)
show("all student answers", everything)
show("passed validators", accepted)

with open(f"/kaggle/working/predictions_{RUN}.jsonl", "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

## Save the adapter

In [ ]:
model.save_pretrained(f"adapter_{RUN}")
tokenizer.save_pretrained(f"adapter_{RUN}")
shutil.make_archive(f"/kaggle/working/adapter_{RUN}", "zip", f"adapter_{RUN}")